# Physics-Informed Loss Sweep — DCRNN

This Kaggle kernel evaluates whether relaxed biological envelope and spatial
regularizers improve forecasting accuracy and resolve the lag-under-reaction
pathology for **DCRNN**.

## Evaluated Arms:
1. `base`: Unconstrained GNN baseline.
2. `envelope`: SEIR-SEI upper biological growth ceiling ($r_{\text{max}} = 2.3884$) and clearance bounds.
3. `spatial`: District-normalized spatial Dirichlet flux smoothness across connected districts.
4. `composite`: Combined biological envelope + spatial smoothness + non-negativity.
5. `outbreak_aware`: Composite regularizer + asymmetric under-prediction weighting ($w_{\text{under}} = 2.5$).

Protocol: 3 chronological origins ($0.55, 0.70, 0.85$) $\times$ 3 seeds ($0, 1, 2$) = 9 runs per arm.
Evaluated on all windows and artifact-free clean windows (excluding the 2019 backlog artifact).

## 1. Environment Bootstrap

In [ ]:
import subprocess, sys, os, json, time, hashlib, re
from pathlib import Path

REPO = "https://github.com/MLOpenSourceOpenScience/disease_modeling_MLOS2.git"
COMMIT = "45f1c0878f002407633ed1237638734faa9ceb2b"
BLOB_NPY = "f7cfa6ec31a4058584fe256a1d6de6800e72a5b1"
BLOB_ADJ = "f3a3cb7f43998850410b0a494f16f331c3830a84"
SEGMENTS = [0.6, 0.7, 0.8, 0.9, 1.0]     # the authors' __main__ segment list

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
# Build outside /kaggle/working: anything left there becomes kernel output, and a
# 40 MB clone plus a venv makes `kaggle kernels output` unusably slow.
SCRATCH = Path("/tmp/repro") if Path("/tmp").exists() else WORK
SCRATCH.mkdir(parents=True, exist_ok=True)
SRC = SCRATCH / "mlos2"
VENV = SCRATCH / "venv311"
PY311 = VENV / "bin" / "python"

os.environ["MPLBACKEND"] = "Agg"          # their plotting helpers call plt.show()


def sh(*args, check=True, quiet=False, **kw):
    """Run a command, echoing it, and abort on a non-zero exit."""
    if not quiet:
        print("$", " ".join(str(a) for a in args))
    r = subprocess.run([str(a) for a in args], text=True, capture_output=True, **kw)
    if r.stdout.strip() and not quiet:
        print(r.stdout[-2000:])
    if r.returncode != 0:
        print(r.stderr[-4000:])
        if check:
            raise SystemExit("command failed: " + " ".join(str(a) for a in args))
    return r


print("kernel python:", sys.version.split()[0])

In [ ]:
# Kaggle runs Python 3.12; torch 2.1.2 has no cp312 wheel. Fetch a standalone
# 3.11 with uv rather than bumping the authors' pinned torch.
sh(sys.executable, "-m", "pip", "install", "-q", "uv")

UV = [sys.executable, "-m", "uv"]
sh(*UV, "python", "install", "3.11")
sh(*UV, "venv", "--python", "3.11", str(VENV))

PIP = [*UV, "pip", "install", "-q", "--python", str(PY311)]

# Exactly the versions in the authors' requirements.txt.
sh(*PIP, "torch==2.1.2", "--index-url", "https://download.pytorch.org/whl/cpu")
sh(*PIP, "torch_scatter", "torch_sparse", "-f",
   "https://data.pyg.org/whl/torch-2.1.2+cpu.html")

# Deviation 2: the authors pin torch_geometric==2.5.3, but PGT 0.54.0 imports
# torch_geometric.utils.to_dense_adj, which PyG removed in 2.4. 2.4.0 is the
# newest release where all five architectures import.
sh(*PIP, "torch_geometric==2.4.0", "numpy~=1.26.2", "pandas~=2.2.0",
   "scikit_learn==1.4.0", "statsmodels==0.14.1", "decorator==4.4.2",
   "cython", "matplotlib", "tqdm")

# Deviation 1: PGT's own pandas<=1.3.5 pin contradicts the authors'
# pandas~=2.2.0 and has no Python 3.11 wheel.
sh(*PIP, "--no-deps", "torch_geometric_temporal==0.54.0")

In [ ]:
probe = sh(str(PY311), "-c", """
import json, torch, torch_geometric, pandas, numpy
from torch_geometric_temporal import A3TGCN, ASTGCN, AAGCN
from torch_geometric_temporal.nn.recurrent import DCRNN
from torch_geometric_temporal.signal import StaticGraphTemporalSignal, temporal_signal_split
print(json.dumps({
    "python": ".".join(map(str, __import__("sys").version_info[:3])),
    "torch": torch.__version__,
    "torch_geometric": torch_geometric.__version__,
    "pandas": pandas.__version__,
    "numpy": numpy.__version__,
}))
""", quiet=True)

versions = json.loads(probe.stdout.strip().splitlines()[-1])
print(json.dumps(versions, indent=2))
assert versions["python"].startswith("3.11"), versions["python"]
assert versions["torch"].startswith("2.1.2"), versions["torch"]
assert versions["torch_geometric"] == "2.4.0", versions["torch_geometric"]
print("all five architectures import OK under Python 3.11")

## 2. Clone Repository with Physics Framework

In [ ]:
ARCH = "DCRNN"
PROJECT = "https://github.com/rathishTharusha/dengue-forecasting-gnn.git"
BRANCH = "feat/physics-informed-loss"
PROJ = SCRATCH / "project"

if not PROJ.exists():
    sh("git", "clone", "--depth", "1", "--branch", BRANCH, PROJECT, str(PROJ))
print("Cloned branch:", BRANCH)

RUNNER = PROJ / "analysis" / "_build" / "run_physics_experiments.py"
assert RUNNER.exists(), f"Runner script not found at {RUNNER}"
RESULTS_JSON = WORK / f"physics_envelope_{ARCH}.json"

## 3. Execute Physics Sweep for Architecture

In [ ]:
started = time.time()
print(f"Executing physics sweep for {ARCH} (20 seeds x 3 origins x 2 arms = 120 runs)...")

proc = subprocess.Popen(
    [
        str(PY311),
        "-u",
        str(RUNNER),
        "--arch",
        ARCH,
        "--arms",
        "base",
        "spatial",
        "--seeds",
        "20",
        "--out-dir",
        str(WORK),
    ],
    cwd=str(PROJ),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()

assert proc.returncode == 0, f"Physics sweep runner failed with exit code {proc.returncode}"
print(f"\nSweep for {ARCH} completed successfully in {(time.time()-started)/60:.1f} min")

## 4. Empirical Evaluation & Comparison Tables

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

assert RESULTS_JSON.exists(), f"Results file {RESULTS_JSON} does not exist!"
records = json.loads(RESULTS_JSON.read_text(encoding="utf-8"))
df = pd.DataFrame(records)

print(f"=== SUMMARY RESULTS FOR {ARCH} (n=9 runs per arm) ===")
p_floor_clean = df[df["arch"] == "persistence"]["RMSE_clean"].mean()
p_floor_all = df[df["arch"] == "persistence"]["RMSE"].mean()

print(f"Persistence Floor: All RMSE = {p_floor_all:.2f}, Clean RMSE = {p_floor_clean:.2f}\n")

model_df = df[df["arch"] == ARCH]
summary = model_df.groupby("increment").agg({
    "RMSE": ["mean", "std"],
    "RMSE_clean": ["mean", "std"],
    "MAE_clean": ["mean", "std"],
    "growth_max": ["mean"],
    "growth_p99": ["mean"]
})
print(summary)

In [ ]:
print(f"\n=== PAIRED DIFFERENCES AGAINST UNCONSTRAINED BASE ({ARCH}) ===")
base_clean = model_df[model_df["increment"] == "base"].set_index(["origin", "seed"])["RMSE_clean"]

for arm in ["envelope", "spatial", "composite", "outbreak_aware"]:
    sub = model_df[model_df["increment"] == arm].set_index(["origin", "seed"])["RMSE_clean"]
    common = sorted(set(base_clean.index) & set(sub.index))
    if common:
        diffs = sub.loc[common] - base_clean.loc[common]
        _, pval = stats.ttest_rel(sub.loc[common], base_clean.loc[common])
        print(f"  {arm:15s} dRMSE_clean: {diffs.mean():+6.2f} (better in {(diffs < 0).sum()}/{len(common)}, p={pval:.4f})")

In [ ]:
# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

arms = ["base", "envelope", "spatial", "composite", "outbreak_aware"]
clean_means = [model_df[model_df["increment"] == a]["RMSE_clean"].mean() for a in arms]
gmax_means = [model_df[model_df["increment"] == a]["growth_max"].mean() for a in arms]

# Panel 1: Clean RMSE
bars = ax1.bar(arms, clean_means, color=["#7f7f7f", "#1f77b4", "#2ca02c", "#ff7f0e", "#d62728"], alpha=0.85)
ax1.axhline(p_floor_clean, color="red", linestyle="--", label=f"Persistence Floor ({p_floor_clean:.2f})")
ax1.set_ylabel("Clean RMSE (Lower is better)")
ax1.set_title(f"{ARCH}: Artifact-Free Clean RMSE")
ax1.legend()
ax1.grid(axis="y", linestyle=":", alpha=0.6)
ax1.tick_params(axis="x", rotation=20)

# Panel 2: Max Growth Rate
ax2.bar(arms, gmax_means, color=["#7f7f7f", "#1f77b4", "#2ca02c", "#ff7f0e", "#d62728"], alpha=0.85)
ax2.set_ylabel("Max Log-Growth Rate (Higher = more responsive)")
ax2.set_title(f"{ARCH}: Outbreak Growth Responsiveness")
ax2.grid(axis="y", linestyle=":", alpha=0.6)
ax2.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()